# ITP4256 AI Application Studies — Lab 04 (CA2 · 10%)
## Hong Kong Used Car Price Estimator — Motor City Prototype

---

### Scenario — you work at Hong Kong Motor City

Your manager wants a **Hong Kong used-car HKD price tool** for the showroom floor. A colleague should type in a car's details — **brand, year, mileage, horsepower (PS)**, and any other specs you can justify — and get an **estimated resale price in HKD**, faster and more consistent than guessing.

### Your mission

1. Train a **real** regression model for `Price_HKD` on the real Motor City transaction CSV.
2. Compare three numbers on the **same test set**: a dummy **baseline** (always guess the average price), the weak **starter**, and **your AI-improved** model.
3. Put the model behind a Streamlit page the manager can click.
4. Deploy it so it has a public `*.streamlit.app` address.

This prototype is a **quote ballpark** (a talking range for staff). It is **not** an official valuation form (a bank / insurer / stamped figure you would be legally standing behind).

### Today's journey
```
Real HK transaction data (provided) → Google Colab (clean + train) → AI improve(chat tool) → GitHub (upload 3 files) → Streamlit Community Cloud (deploy live) → Moodle (submit)
```

### How this lab works (different from previous weeks!)
1. Most weeks you only changed a `# CHANGE ME` knob. **Today the starter is deliberately weak — you use an AI Copilot (ChatGPT / Gemini / Claude) to improve it.**
2. Prompts are an **English skeleton**. Copy it, **fill every blank**, then send. Paste the **filled** skeleton onto the worksheet (this is graded). Do **not** send a Chinese prompt.
3. **Keep the same variable names** (`FEATURES`, `X_all`, `y_all`, `model`) when you improve things.
4. Every filled prompt + bug + fix goes in your worksheet **AI Log**.

### Fixed settings (do not change)
- `TEST_SIZE = 0.2` · `RANDOM_STATE = STUDENT_ID_LAST4` (your own Student ID, set in Task 0)
- Use the provided real dataset only — no synthetic/fake data.


## Task 0


In [ ]:
# Load tools we need today
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# =========================
# CHANGE ME — Your details
# =========================
STUDENT_NAME = "YOUR NAME HERE"
STUDENT_ID = " "          # e.g. "12345678"
STUDENT_ID_LAST4 = int(STUDENT_ID[-4:]) if STUDENT_ID.strip().isdigit() else 1234

print(f"Student: {STUDENT_NAME} | ID: {STUDENT_ID}")
print("Keep this cell's output visible — it is your submission watermark.")

TEST_SIZE = 0.2
RANDOM_STATE = STUDENT_ID_LAST4
print("\nTEST_SIZE =", TEST_SIZE, "| RANDOM_STATE =", RANDOM_STATE)
print("Do not change TEST_SIZE or RANDOM_STATE.")


## Task 1 — Data Source

Today's data comes from a real Hong Kong car dealership website's public "Transaction Reference" page:

https://motorcity.hk/en/transaction-reference

Open that link and skim a page or two. On **worksheet Part 1**, write: the business problem in 1 sentence, and which columns you can see there (Brand, Model, Manufacture Year, Mileage, Price...).


In [ ]:
# Path to the CSV on your Drive — change only if your folder name is different
CSV_PATH = "Lab04_hk_car_price.csv"

df = pd.read_csv(CSV_PATH)
TARGET = "Price_HKD"

print("Loaded:", CSV_PATH)
print("Source: https://motorcity.hk/en/transaction-reference (real HK dealer transaction data)")
print("shape:", df.shape)
df.head()


## Task 2 — Explore (fill worksheet Part 2)

Run the cell below. On your worksheet, write down: `shape`, whether there are missing values, and which column(s) you would NOT use as a feature.


In [ ]:
print("shape:", df.shape)
print("\ncolumns:", list(df.columns))
print("\nTransaction date range:", df["Transaction_Date"].min(), "→", df["Transaction_Date"].max())
missing = df.isnull().sum()
print("\nmissing values per column (0 rows shown = none missing):\n", missing[missing > 0])
print("\nPrice_HKD summary:\n", df["Price_HKD"].describe())
print("\nTop 10 brands:\n", df["Brand"].value_counts().head(10))
print("\nMedian Price_HKD: HK${:,.0f}".format(df["Price_HKD"].median()))


## Task 3 — Data cleaning (fill worksheet Part 3)

Empty cells are not automatically “broken data”. Decide **what you will do** before you add columns to the model.

You can decide yourself.


In [ ]:
# Cleaning workspace. Later tasks use df unless you set df = df_clean.
# Set CHOICE to one of: "skip", "drop", "fill0"
CHOICE = "____"

print("Missing values:\n", df.isnull().sum())
print("\nDisplacement_cc empty rows:", int(df["Displacement_cc"].isna().sum()))
print("Other columns empty? (should be 0 for most):")
print(df.drop(columns=["Displacement_cc"]).isnull().sum().sum(), "empty cells outside Displacement_cc")

if CHOICE == "____":
    raise ValueError("Set CHOICE to skip, drop, or fill0, then run this cell again.")

if CHOICE == "skip":
    df_clean = df.copy()
elif CHOICE == "drop":
    df_clean = df.dropna(subset=["Displacement_cc"])
elif CHOICE == "fill0":
    df_clean = df.copy()
    df_clean["Displacement_cc"] = df_clean["Displacement_cc"].fillna(0)
else:
    raise ValueError("CHOICE must be skip, drop, or fill0.")

print("\nRows in df:", len(df), "| rows in df_clean:", len(df_clean))
print("If you want later tasks to use the cleaned table, run:  df = df_clean")


## Task 4 — Baseline vs the weak starter (fill worksheet Part 4)

Run this cell **as-is first**. It trains a real model (`model.fit`) but only uses year + mileage — **deliberately weak**.

- **Baseline** = always guess the average `Price_HKD` on the training fold.
- **Starter** = `LinearRegression` on `Manufacture_Year` + `Mileage_km` only.


In [ ]:
# ============================================================
# STARTER — runs, but is DELIBERATELY weak. Do not "fix" this cell.
# Task 5 is where you improve. Keep FEATURES / X_all / y_all / model names in Task 5.
# ============================================================

STARTER_FEATURES = ["Manufacture_Year", "Mileage_km"]


# X = inputs (year and mileage). y = the real price we want to predict.
X_all = df[STARTER_FEATURES]
y_all = df[TARGET]

# Keep 80% of rows to train. Hold out 20% as the test set.
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE)

# Baseline: guess the training-set average price for every test car.
baseline_pred = [y_train.mean()] * len(y_test)
BASELINE_MAE = mean_absolute_error(y_test, baseline_pred)

# Starter model: learn one straight line from year and mileage.
starter_model = LinearRegression()
starter_model.fit(X_train, y_train)

# Train MAE = error on cars it has seen. Test MAE = error on cars it has not seen.
STARTER_TRAIN_MAE = mean_absolute_error(y_train, starter_model.predict(X_train))
STARTER_TEST_MAE = mean_absolute_error(y_test, starter_model.predict(X_test))
STARTER_TEST_R2 = r2_score(y_test, starter_model.predict(X_test))

print(f"Training average price (the guess): HK${y_train.mean():,.0f}")
print(f"Baseline Test MAE: HK${BASELINE_MAE:,.0f}")
print(f"Starter features: {STARTER_FEATURES}")
print(f"Starter Train MAE: HK${STARTER_TRAIN_MAE:,.0f}")
print(f"Starter Test MAE:  HK${STARTER_TEST_MAE:,.0f}   <- main number (smaller = better)")
print(f"Starter Test R²:   {STARTER_TEST_R2:.2f}   (1=perfect, 0=like guessing the mean)")
print(f"\nLift vs baseline: {(1 - STARTER_TEST_MAE / BASELINE_MAE):.1%}  <- probably tiny right now!")

## Task 5 — AI Improve the model + compare (fill worksheet Part 5)

Your manager asked for brand / year / mileage / PS — the **starter does not use most of those yet**. Improve the model so **Test MAE is clearly lower than the starter**.

### Prompt skeleton (copy → fill every blank in English → send)

Do **not** send the skeleton with empty blanks. Paste the **filled** version onto the worksheet.

```
I am a student in Google Colab.
I already have a pandas DataFrame called df with real Hong Kong used-car sales.
The target column is Price_HKD (Hong Kong dollars).
pandas and scikit-learn are already imported.
Please give Python code for ONE Colab cell that I can paste.
Comment every line in simple English.
Do not write a full .py file.

My current FEATURES = [ ________ , ________ ]
I am using LinearRegression.
My starter Test MAE is about HK$ ________
(smaller MAE is better)

Please help me improve Test MAE.
I want to add this column / these columns: ________
because: ________

Keep the variable names FEATURES, X_all, y_all, model.
Do not change TEST_SIZE or RANDOM_STATE.
Print train MAE and test MAE in HKD.
If a column has empty cells, handle them so fit() does not crash.
Comment every line — I must explain this myself.

Optional extra (only if I want it): ________
```

Then replace the starter block in the next cell, run it, and run the comparison cell.


In [ ]:
# ============================================================
# IMPROVED MODEL — replace this block with YOUR AI code.
# Keep FEATURES / X_all / y_all / model. Do not change TEST_SIZE / RANDOM_STATE.
#
# Use these print lines exactly:
# print("Features used:", FEATURES)
# print(f"Improved Train MAE: HK${train_mae:,.0f}")
# print(f"Improved Test MAE:  HK${test_mae:,.0f}")
# print(f"Improved Test R²:   {test_r2:.2f}")
# ============================================================

# Change Here==============================================================================
FEATURES = ["Manufacture_Year", "Mileage_km"]

X_all = df[FEATURES]
y_all = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

model = LinearRegression()
model.fit(X_train, y_train)

train_mae = mean_absolute_error(y_train, model.predict(X_train))
test_mae = mean_absolute_error(y_test, model.predict(X_test))
test_r2 = r2_score(y_test, model.predict(X_test))

print("Features used:", FEATURES)
print(f"Improved Train MAE: HK${train_mae:,.0f}")
print(f"Improved Test MAE:  HK${test_mae:,.0f}")
print(f"Improved Test R²:   {test_r2:.2f}")
print("\nIf Test MAE is still almost equal to the starter, you have not improved yet.")
# Change Here==============================================================================


In [ ]:
# Comparison — needs Task 4 AND Task 5 to have been run.
print("=== Comparison (smaller Test MAE = better) ===")
print(f"{'Version':<22} {'Test MAE':>14}  note")
print(f"{'Baseline':<22} HK${BASELINE_MAE:>10,.0f}  always guess the average")
print(f"{'Starter':<22} HK${STARTER_TEST_MAE:>10,.0f}  {STARTER_FEATURES}")
print(f"{'Improved':<22} HK${test_mae:>10,.0f}  {FEATURES}")
print(f"\nTrain MAE improved: HK${train_mae:,.0f}   |  Test R² improved: {test_r2:.2f}")
if train_mae < test_mae * 0.6:
    print("Watch the gap: Train MAE much smaller than Test MAE can mean overfitting (memorising).")


## Task 6 — Evaluation chart (fill worksheet Part 6)
A MAE table is one number. Pick **ONE** chart that can convince a colleague how far the quote can be trusted.
1. **A. Predicted vs actual on the TEST set**, with a **y = x** line. Points along the diagonal, from bottom-left to top-right, follow the real price. A cluster in the corner only means many cheap cars.
2. **B. Train MAE vs Test MAE bars** for baseline, starter, and your improved model. Use this to check overfitting.

Do **not** use a heatmap, a price histogram, or the training rows to claim the model is accurate.

Fill every blank. Send the skeleton to the AI. Paste the code into the next cell and run it.


### Prompt skeleton(fill every blank before you send)

```
I am a student in Google Colab.
I already have X_test, y_test, and a trained model called model.
I also have these values in HKD: BASELINE_MAE, STARTER_TRAIN_MAE, STARTER_TEST_MAE, train_mae, test_mae.
pandas, matplotlib, and scikit-learn are already imported.
Please give Python code for ONE Colab cell that I can paste.
Comment every line in simple English.
Do not write a full .py file.

I will plot: ________
because this chart shows: ________
I will not plot: ________
because that would only show: ________

Use the test set only if I chose predicted vs actual.
Do not plot the training rows for that chart.
Do not draw a heatmap.
Label the axes in HKD.
```


In [ ]:
# Paste your ONE chart cell below, then run.


## Task 8 — Cross the platform: GitHub → Streamlit Community Cloud (fill worksheet Part 7)

Give your prototype a **permanent public address**.

1. Run the 2 cells below to write `app.py` and `requirements.txt`. In `app.py`, **change the `FEATURES` line** to your own column names. You may include `Brand`. Do not add `Displacement_cc`.
2. Run the 3rd cell to **download 3 files**: `app.py`, `requirements.txt`, **and the CSV**.
3. https://github.com/new → **public** repo → **uploading an existing file** → drag in **all 3 files** (same folder, not a subfolder) → Commit.
4. https://share.streamlit.io → Sign in with GitHub → Create app → repo / `main` / `app.py` → Deploy.
5. Wait 1–2 minutes. Open `xxx.streamlit.app`. Copy the URL to the worksheet.

**Note.** If Cloud is down, write what happened on the worksheet.


In [ ]:
%%writefile app.py
# This file must work standing ALONE on Streamlit Cloud (no Colab, no Drive).
# Upload Lab04_hk_car_price.csv in the SAME GitHub folder as this file.

import pandas as pd
import streamlit as st
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# CHANGE THIS LINE. Use your own column names. Do not leave the words PASTE / HERE.
# Do not add Displacement_cc. Empty engine cc will crash training.
FEATURES = ['paste', 'your', 'feature', 'here']         # Example: ["Manufacture_Year", "Mileage_km"]
RANDOM_STATE = 42   # public demo — does not need to match your Student ID

@st.cache_data
def load_and_train():
    df = pd.read_csv("Lab04_hk_car_price.csv")
    X_all = df[FEATURES].copy()
    y_all = df["Price_HKD"]

    # Brand is text. Split it into number columns before fit().
    text_cols = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(X_all[c])]
    if text_cols:
        X_all = pd.get_dummies(X_all, columns=text_cols, drop_first=True)
        X_all = X_all.astype(float)

    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, random_state=RANDOM_STATE
    )
    model = LinearRegression()
    model.fit(X_train, y_train)
    return df, model, list(model.feature_names_in_)

df, model, model_columns = load_and_train()

st.title("HK Used Car Price Estimator — YourName_SID_CA2 Prototype")                  #<- Change your name here!!!=======================================
st.write("Predicts **resale price (HKD)** from real Hong Kong Motor City transactions. This is a quote ballpark — not an official valuation form.")

# Sliders and brand menus are built from FEATURES. Do not delete this loop.
inputs = {}
for col in FEATURES:
    if not pd.api.types.is_numeric_dtype(df[col]):
        inputs[col] = st.selectbox(col, sorted(df[col].dropna().unique().tolist()))
    else:
        inputs[col] = st.slider(
            col, float(df[col].min()), float(df[col].max()), float(df[col].mean())
        )

if st.button("Estimate Price"):
    row = pd.DataFrame([inputs])
    text_cols = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        row = pd.get_dummies(row, columns=text_cols)
    row = row.reindex(columns=model_columns, fill_value=0)
    price = model.predict(row)[0]
    st.success(f"Estimated price: HK${price:,.0f}")


In [ ]:
%%writefile requirements.txt
streamlit
pandas
scikit-learn
numpy


In [ ]:
# Local copy of the CSV so it downloads with the other 2 files
df.to_csv("Lab04_hk_car_price.csv", index=False)

from google.colab import files

files.download("app.py")
files.download("requirements.txt")
files.download("Lab04_hk_car_price.csv")
print("Got all 3 files,Then go github")
